<a href="https://colab.research.google.com/github/AAwaisYaseen/ai-avatar-communication-/blob/main/COMP6011_LLM_PoC_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate torch scikit-learn
!pip install -q bitsandbytes>=0.46.1

In [ ]:
import bitsandbytes as bnb
print(f"bitsandbytes version: {bnb.__version__}")

bitsandbytes version: 0.49.2


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
from sklearn.metrics import accuracy_score, f1_score, classification_report

HF_TOKEN = userdata.get('HF_TOKEN')
print("Token loaded.")

Token loaded.


In [3]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

df = pd.read_excel('/content/drive/MyDrive/AAIRT-Task2/student_assignment_10_cases.xlsx',
                   sheet_name='Assignment_Cases')

print(f"Loaded {len(df)} cases successfully")
print(df[['Case ID', 'Risk Level']])

Mounted at /content/drive
Loaded 10 cases successfully
   Case ID Risk Level
0        1    attempt
1        2    attempt
2        3   behavior
3        4   behavior
4        5   ideation
5        6   ideation
6        7  indicator
7        8  indicator
8        9       safe
9       10       safe


In [ ]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16
# )

# mistral_model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# print("Loading Mistral 7B tokenizer...")
# mistral_tokenizer = AutoTokenizer.from_pretrained(mistral_model_id)

# print("Loading Mistral 7B model...")
# mistral_model = AutoModelForCausalLM.from_pretrained(
#     mistral_model_id,
#     quantization_config=bnb_config,
#     device_map="auto"
# )

# print("Mistral 7B loaded successfully.")

Loading Mistral 7B tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Loading Mistral 7B model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
mistral_model_id = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

print("Loading Mistral 7B tokenizer...")
mistral_tokenizer = AutoTokenizer.from_pretrained(mistral_model_id)

print("Loading Mistral 7B model...")
mistral_model = AutoModelForCausalLM.from_pretrained(
    mistral_model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Mistral 7B loaded successfully.")

Loading Mistral 7B tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

Mistral 7B loaded successfully.


In [ ]:
def classify_risk(model, tokenizer, conversation):

    prompt = f"""Read the conversation and classify the suicide risk level.

Definitions:
- attempt: person has made or is actively planning a suicide attempt
- behavior: person is doing things suggesting they may attempt suicide soon
- ideation: person is having thoughts of suicide but no clear plan
- indicator: person is showing serious distress like hopelessness but no direct suicidal thoughts
- safe: no signs of suicide risk

Conversation:
{conversation}

Reply with one word only. One word. No sentences. No explanation. Just the label:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

    valid_labels = ["attempt", "behavior", "ideation", "indicator", "safe"]
    for label in valid_labels:
        if label in response:
            return label

    return "unknown"

Function defined.


In [ ]:
print("Running Mistral 7B on 10 cases...\n")

mistral_predictions = []

for index, row in df.iterrows():
    case_id = row['Case ID']
    conversation = row['Paraphrased Dialogue']
    ground_truth = row['Risk Level']

    predicted = classify_risk(mistral_model, mistral_tokenizer, conversation)
    mistral_predictions.append(predicted)

    print(f"Case {case_id} | Ground Truth: {ground_truth} | Predicted: {predicted}")

print("\nDone. All 10 cases.")

Running Mistral 7B on all 10 cases...

Case 1 | Ground Truth: attempt | Predicted: attempt
Case 2 | Ground Truth: attempt | Predicted: attempt
Case 3 | Ground Truth: behavior | Predicted: indicator
Case 4 | Ground Truth: behavior | Predicted: attempt
Case 5 | Ground Truth: ideation | Predicted: attempt
Case 6 | Ground Truth: ideation | Predicted: indicator
Case 7 | Ground Truth: indicator | Predicted: indicator
Case 8 | Ground Truth: indicator | Predicted: indicator
Case 9 | Ground Truth: safe | Predicted: safe
Case 10 | Ground Truth: safe | Predicted: indicator

Done. All 10 cases.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

ground_truth = list(df['Risk Level'])

# Llama results.
llama_predictions = ['ideation', 'attempt', 'attempt', 'indicator',
                     'indicator', 'attempt', 'ideation', 'attempt',
                     'behavior', 'behavior']

# Mistral results
mistral_predictions = ['attempt', 'attempt', 'indicator', 'attempt',
                       'attempt', 'indicator', 'indicator', 'indicator',
                       'safe', 'indicator']

llama_acc = accuracy_score(ground_truth, llama_predictions)
llama_f1 = f1_score(ground_truth, llama_predictions, average='macro', zero_division=0)

mistral_acc = accuracy_score(ground_truth, mistral_predictions)
mistral_f1 = f1_score(ground_truth, mistral_predictions, average='macro', zero_division=0)

print("Summary = \n")
print(f"Llama 3.1 8B - Accuracy: {llama_acc:.2f} | Macro F1: {llama_f1:.2f}")
print(f"Mistral 7B - Accuracy: {mistral_acc:.2f} | Macro F1: {mistral_f1:.2f}")

=== Summary ===

Llama 3.1 8B - Accuracy: 0.10 | Macro F1: 0.07
Mistral 7B - Accuracy: 0.50 | Macro F1: 0.38
